In [ ]:
# --- Act 0: Setup (hidden) ---
import warnings
warnings.filterwarnings('ignore')

import html
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from textwrap import shorten

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats
from IPython.display import HTML, display, Markdown

# --- Path resolution ---
_here = Path('.').resolve()
if (_here / 'projects').exists():
    REPO_ROOT = _here
elif (_here.parent / 'projects').exists():
    REPO_ROOT = _here.parent
else:
    raise FileNotFoundError(
        "Cannot find 'projects/' directory. "
        "Run this notebook from the repository root or the notebooks/ directory."
    )
CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026'

# --- Plotly config ---
pio.renderers.default = "notebook+png"
pio.templates.default = "plotly_white"

# --- Seaborn / matplotlib theme ---
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

# --- Color palette ---
ENTRY_IDS = [
    'LLM01', 'LLM02', 'LLM03', 'LLM04', 'LLM05',
    'LLM06', 'LLM07', 'LLM08', 'LLM09', 'LLM10',
    'NEW-ITSCD', 'NEW-MA', 'NEW-MSDA', 'NEW-MTIE', 'NEW-PMP', 'NEW-WLA',
    'ROLL-CFAS', 'ROLL-CMSB', 'ROLL-LAPTF', 'ROLL-SICG',
]
FRAME_BLIND = {'LLM04', 'LLM08', 'LLM10'}

_incumbent_blues = sns.color_palette('mako', n_colors=12)
_new_oranges = sns.color_palette('flare', n_colors=8)
_roll_purples = sns.color_palette('crest', n_colors=6)
ENTRY_COLORS = {}
_ib = 0; _in = 0; _ir = 0
for eid in ENTRY_IDS:
    if eid in FRAME_BLIND:
        ENTRY_COLORS[eid] = '#999999'
    elif eid.startswith('LLM'):
        ENTRY_COLORS[eid] = _incumbent_blues[_ib % len(_incumbent_blues)]
        _ib += 1
    elif eid.startswith('NEW'):
        ENTRY_COLORS[eid] = _new_oranges[_in % len(_new_oranges)]
        _in += 1
    elif eid.startswith('ROLL'):
        ENTRY_COLORS[eid] = _roll_purples[_ir % len(_roll_purples)]
        _ir += 1

# --- Deep-dive sidebar helper ---
def sidebar(title, content):
    """Render a collapsible deep-dive sidebar."""
    return HTML(
        f'<details style="margin: 1em 0; padding: 0.5em; '
        f'border-left: 3px solid #4a86c8; background: #f8f9fa;">'
        f'<summary style="cursor: pointer; font-weight: bold; '
        f'color: #2c5282;">{title}</summary>'
        f'<div style="margin-top: 0.8em; line-height: 1.6;">{content}</div>'
        f'</details>'
    )

def callout_box(text, border_color='#e53e3e', bg_color='#fff5f5'):
    """Render an always-visible boxed callout (not collapsed)."""
    return HTML(
        f'<div style="margin: 1em 0; padding: 1em; border-left: 4px solid {border_color}; '
        f'background: {bg_color}; line-height: 1.6;">{text}</div>'
    )

# --- Load all data ---
DATA = {}

with open(CYCLE / 'prereg' / 'rubric.json') as f:
    DATA['rubric'] = json.load(f)

with open(CYCLE / 'classify' / 'labeled_incidents_multimodel.json') as f:
    DATA['incidents'] = json.load(f)

prelabels = []
with open(CYCLE / 'calibration' / 'llm_prelabels.jsonl') as f:
    for line in f:
        prelabels.append(json.loads(line))
DATA['prelabels'] = prelabels

goldset = []
with open(CYCLE / 'calibration' / 'adjudicated_goldset.jsonl') as f:
    for line in f:
        goldset.append(json.loads(line))
DATA['goldset'] = goldset

precision_verif = []
with open(CYCLE / 'calibration' / 'precision_verification.jsonl') as f:
    for line in f:
        precision_verif.append(json.loads(line))
DATA['precision_verification'] = precision_verif

with open(CYCLE / 'calibration' / 'posteriors.json') as f:
    DATA['posteriors'] = json.load(f)

with open(CYCLE / 'calibration' / 'diagnostic.json') as f:
    DATA['diagnostic'] = json.load(f)

with open(CYCLE / 'infer' / 'inference_summary.json') as f:
    DATA['inference_summary'] = json.load(f)

DATA['lambda_samples'] = np.load(CYCLE / 'infer' / 'lambda_samples.npy')

with open(CYCLE / 'results' / 'concordance.json') as f:
    DATA['concordance'] = json.load(f)

with open(CYCLE / 'results' / 'selection_bias.json') as f:
    DATA['selection_bias'] = json.load(f)

with open(CYCLE / 'results' / 'rank_comparison_report.md') as f:
    DATA['rank_comparison_md'] = f.read()

# Build lookup tables
ENTRY_NAMES = {e['entry_id']: e['canonical_name'] for e in DATA['rubric']['entries']}
INFER_ENTRY_ORDER = DATA['inference_summary']['entry_ids']

# Validate shapes
assert DATA['lambda_samples'].shape == (16000, 20), (
    f"Expected lambda_samples shape (16000, 20), got {DATA['lambda_samples'].shape}"
)
assert len(INFER_ENTRY_ORDER) == 20, (
    f"Expected 20 entry_ids in inference_summary, got {len(INFER_ENTRY_ORDER)}"
)

print(f"Loaded {len(DATA)} data files. {len(DATA['incidents'])} incidents, "
      f"{len(DATA['prelabels'])} prelabels, {len(DATA['goldset'])} adjudications, "
      f"{DATA['lambda_samples'].shape[0]} posterior draws. Ready.")